# **Makemore: Multi-Layer Perceptrons**
Scales the ideas from Micrograd and Makemore: Bigrams into a true multi-layer neural network with a hidden layer and non-linear activation functions. Model moves past and beyond static counting. Introduces data splitting, hyperparameter tuning, under and overfitting, etc. 

**Paper Followed:** A Neural Probabilistic Language Model - Bengio et al. 2003

## **Building Data-Set**

In [4]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

In [1]:
# read in all the words
words = open('names.txt', 'r').read().splitlines()
words[:8]

['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']

In [5]:
# build vocab of characters and mapping to/from integers
chars = sorted(list(set(''.join(words))))
stoi= {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
print(itos)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


In [6]:
block_size = 3 # context length: how many chars do we take to predict the next one
X, Y = [], []
for w in words[:5]:

    print(w)
    context = [0] * block_size
    for ch in w + '.':
        ix = stoi[ch]
        X.append(context)
        Y.append(ix)
        print(''.join(itos[i] for i in context), '--->', itos[ix])
        context = context[1:] + [ix]

X = torch.tensor(X)
Y = torch.tensor(Y)

emma
... ---> e
..e ---> m
.em ---> m
emm ---> a
mma ---> .
olivia
... ---> o
..o ---> l
.ol ---> i
oli ---> v
liv ---> i
ivi ---> a
via ---> .
ava
... ---> a
..a ---> v
.av ---> a
ava ---> .
isabella
... ---> i
..i ---> s
.is ---> a
isa ---> b
sab ---> e
abe ---> l
bel ---> l
ell ---> a
lla ---> .
sophia
... ---> s
..s ---> o
.so ---> p
sop ---> h
oph ---> i
phi ---> a
hia ---> .


## **Embedding Look-Up Table**
Like the Weight matrix

In [ ]:
# each 27 chars will have 2 dimensional embedding
# gives each character a (x, y) coordinate pair
C = torch.randn((27, 2))

In [9]:
C[5]

# Under the hood:
# F.one_hot(torch.tensor(5), num_classes=27).float() @ C

tensor([0.1402, 1.3718])

In [10]:
C[[5, 6, 7]]

tensor([[ 0.1402,  1.3718],
        [ 2.1859,  0.2449],
        [ 2.3637, -0.9832]])

In [16]:
emb = C[X]
emb.shape

torch.Size([32, 3, 2])

## **Hidden Layer**

In [14]:
W1 = torch.randn((6,100))
b1 = torch.randn(100)

In [18]:
torch.cat([emb[:, 0, :], emb[:, 1, :], emb[:, 2, :]], 1)

tensor([[-1.0268,  0.5185, -1.0268,  0.5185, -1.0268,  0.5185],
        [-1.0268,  0.5185, -1.0268,  0.5185,  0.1402,  1.3718],
        [-1.0268,  0.5185,  0.1402,  1.3718,  0.0151,  1.2758],
        [ 0.1402,  1.3718,  0.0151,  1.2758,  0.0151,  1.2758],
        [ 0.0151,  1.2758,  0.0151,  1.2758,  0.0973, -0.7330],
        [-1.0268,  0.5185, -1.0268,  0.5185, -1.0268,  0.5185],
        [-1.0268,  0.5185, -1.0268,  0.5185, -0.6652, -0.5247],
        [-1.0268,  0.5185, -0.6652, -0.5247, -0.5995, -0.7770],
        [-0.6652, -0.5247, -0.5995, -0.7770,  0.4544, -0.6191],
        [-0.5995, -0.7770,  0.4544, -0.6191, -2.0248,  0.1817],
        [ 0.4544, -0.6191, -2.0248,  0.1817,  0.4544, -0.6191],
        [-2.0248,  0.1817,  0.4544, -0.6191,  0.0973, -0.7330],
        [-1.0268,  0.5185, -1.0268,  0.5185, -1.0268,  0.5185],
        [-1.0268,  0.5185, -1.0268,  0.5185,  0.0973, -0.7330],
        [-1.0268,  0.5185,  0.0973, -0.7330, -2.0248,  0.1817],
        [ 0.0973, -0.7330, -2.0248,  0.1